# ⚡ GPU Image Processing & Performance Optimization Engine
### Comparing CPU and GPU Parallel Computing Performance Using CUDA C++

**How to use this notebook:**
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Click **Runtime → Run all**  (or run cells one by one with Shift+Enter)

This notebook demonstrates:
- CUDA C++ kernel execution via PyCUDA
- CPU vs GPU performance benchmarking
- Grayscale conversion, Gaussian Blur, Sobel Edge Detection, and Image Sharpening
- Real speedup measurements on a free Google Colab T4 GPU

## Cell 1 — Install dependencies & verify GPU

In [ ]:
# Install PyCUDA (CUDA Python bridge) and OpenCV
# PyCUDA lets us write and compile real CUDA C++ kernels directly from Python
!pip install pycuda opencv-python-headless --quiet

import subprocess, sys

# Verify CUDA compiler is available
print('=== CUDA Compiler ===')
!nvcc --version

# Verify GPU is assigned
print('\n=== GPU Info ===')
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader

print('\n=== Python packages ===')
import numpy, cv2, pycuda.driver as cuda, pycuda.autoinit
print(f'NumPy  : {numpy.__version__}')
print(f'OpenCV : {cv2.__version__}')
print(f'PyCUDA : OK — device count = {cuda.Device.count()}')
dev = cuda.Device(0)
print(f'GPU    : {dev.name()}')
print(f'VRAM   : {dev.total_memory() // 1024**2} MB')
print(f'Compute: {dev.compute_capability()}')

## Cell 2 — CPU implementations (reference + baseline timing)

In [ ]:
"""
CPU implementations of all four image-processing algorithms.

These serve two purposes:
  1. Correctness reference — GPU output is compared against these.
  2. Baseline timing — speedup = CPU_time / GPU_kernel_time.

All functions operate on NumPy uint8 arrays.
RGB layout: shape (H, W, 3)  |  Grayscale layout: shape (H, W)
"""

import numpy as np
import cv2
import time

# ─────────────────────────────────────────────────────────────────────────────
# Grayscale Conversion
# Formula: Gray = 0.299*R + 0.587*G + 0.114*B  (ITU-R BT.601 luminance)
# Green has the highest weight because human vision is most sensitive to green.
# ─────────────────────────────────────────────────────────────────────────────
def cpu_grayscale(img_rgb: np.ndarray) -> np.ndarray:
    r = img_rgb[:,:,0].astype(np.float32)
    g = img_rgb[:,:,1].astype(np.float32)
    b = img_rgb[:,:,2].astype(np.float32)
    return np.clip(0.299*r + 0.587*g + 0.114*b, 0, 255).astype(np.uint8)

# ─────────────────────────────────────────────────────────────────────────────
# Gaussian Blur
# Each output pixel = weighted average of its neighbourhood.
# Gaussian weight: G(x,y) = exp(-(x²+y²) / (2σ²))
# OpenCV's GaussianBlur is an optimised CPU implementation.
# ─────────────────────────────────────────────────────────────────────────────
def cpu_gaussian_blur(img_rgb: np.ndarray, radius: int = 3, sigma: float = 1.5) -> np.ndarray:
    ksize = 2 * radius + 1
    return cv2.GaussianBlur(img_rgb, (ksize, ksize), sigma)

# ─────────────────────────────────────────────────────────────────────────────
# Sobel Edge Detection
# Applies Gx and Gy convolution kernels; magnitude = sqrt(Gx² + Gy²).
# High magnitude = sharp intensity change = edge.
# ─────────────────────────────────────────────────────────────────────────────
def cpu_sobel(img_gray: np.ndarray) -> np.ndarray:
    img_f = img_gray.astype(np.float32)
    gx = cv2.Sobel(img_f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_f, cv2.CV_32F, 0, 1, ksize=3)
    return np.clip(np.sqrt(gx**2 + gy**2), 0, 255).astype(np.uint8)

# ─────────────────────────────────────────────────────────────────────────────
# Image Sharpening
# Kernel: [0,-1,0; -1,5,-1; 0,-1,0]  — amplifies edges.
# Equivalent to: output = input + (input - blurred_input)
# ─────────────────────────────────────────────────────────────────────────────
def cpu_sharpen(img_rgb: np.ndarray) -> np.ndarray:
    kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
    result = cv2.filter2D(img_rgb.astype(np.float32), -1, kernel)
    return np.clip(result, 0, 255).astype(np.uint8)

# ─────────────────────────────────────────────────────────────────────────────
# Timing helper — returns (result, elapsed_ms)
# ─────────────────────────────────────────────────────────────────────────────
def time_cpu(fn, *args, iterations=5, warmup=2):
    """Run fn(*args) with warm-up, return (result, avg_ms)."""
    for _ in range(warmup):
        fn(*args)
    times = []
    result = None
    for _ in range(iterations):
        t0 = time.perf_counter()
        result = fn(*args)
        times.append((time.perf_counter() - t0) * 1000)
    return result, sum(times) / len(times)

print('✅ CPU functions defined.')

## Cell 3 — CUDA GPU kernels (genuine CUDA C++ compiled at runtime)

In [ ]:
"""
CUDA C++ kernel source code.

PyCUDA compiles these strings with nvcc at runtime, producing PTX bytecode
that runs directly on the GPU.  This is genuine CUDA — not a simulation.

KEY CUDA CONCEPTS DEMONSTRATED:
  __global__   — marks a function as a GPU kernel (callable from CPU)
  blockIdx     — which block this thread belongs to (in the grid)
  threadIdx    — this thread's position within its block
  blockDim     — number of threads per block dimension
  gridDim      — number of blocks in the grid

THREAD INDEXING:
  Each thread computes its 2D pixel coordinate:
    x (column) = blockIdx.x * blockDim.x + threadIdx.x
    y (row)    = blockIdx.y * blockDim.y + threadIdx.y
  One thread = one pixel. Thousands run simultaneously.
"""

import pycuda.driver as cuda
import pycuda.autoinit  # initialises CUDA context automatically
from pycuda.compiler import SourceModule
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# All four kernels compiled together into one CUDA module.
# nvcc compiles this at runtime — you can inspect the PTX if you want.
# ─────────────────────────────────────────────────────────────────────────────
CUDA_SOURCE = r"""
#include <math.h>

/* Helper: clamp float to [0, 255] and return as unsigned char */
__device__ static inline unsigned char dev_clamp(float v) {
    return (v < 0.f) ? 0 : (v > 255.f) ? 255 : (unsigned char)v;
}

/* ===================================================================
   KERNEL 1: Grayscale Conversion
   One thread per pixel. Applies luminance formula independently.
   Input:  packed RGB  (width * height * 3 bytes)
   Output: grayscale   (width * height * 1 byte)
   =================================================================== */
__global__ void kernel_grayscale(
    const unsigned char* input,
    unsigned char* output,
    int width, int height)
{
    /* Compute this thread's pixel coordinates */
    int x = blockIdx.x * blockDim.x + threadIdx.x;  /* column */
    int y = blockIdx.y * blockDim.y + threadIdx.y;  /* row    */

    /* Bounds check: the grid may be larger than the image */
    if (x >= width || y >= height) return;

    int rgb_idx  = (y * width + x) * 3;  /* 3 bytes per RGB pixel   */
    int gray_idx =  y * width + x;       /* 1 byte per gray pixel   */

    float r = input[rgb_idx + 0];
    float g = input[rgb_idx + 1];
    float b = input[rgb_idx + 2];

    /* ITU-R BT.601 luminance formula */
    output[gray_idx] = dev_clamp(0.299f*r + 0.587f*g + 0.114f*b);
}

/* ===================================================================
   KERNEL 2: Sobel Edge Detection
   Applies Gx and Gy 3x3 convolution kernels to a grayscale image.
   Magnitude = sqrt(Gx^2 + Gy^2)
   =================================================================== */
__global__ void kernel_sobel(
    const unsigned char* input,
    unsigned char* output,
    int width, int height)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= width || y >= height) return;

    /* Sobel kernel weights — constant for every thread */
    const int Kx[3][3] = {{-1,0,1},{-2,0,2},{-1,0,1}};
    const int Ky[3][3] = {{-1,-2,-1},{0,0,0},{1,2,1}};

    float gx = 0.f, gy = 0.f;

    /* Convolve 3x3 neighbourhood — clamp-to-edge for borders */
    for (int ky = -1; ky <= 1; ++ky) {
        for (int kx = -1; kx <= 1; ++kx) {
            int nx = max(0, min(x+kx, width-1));
            int ny = max(0, min(y+ky, height-1));
            float p = input[ny*width + nx];
            gx += Kx[ky+1][kx+1] * p;
            gy += Ky[ky+1][kx+1] * p;
        }
    }
    output[y*width + x] = dev_clamp(sqrtf(gx*gx + gy*gy));
}

/* ===================================================================
   KERNEL 3: Image Sharpening
   Laplacian-based sharpening kernel: [0,-1,0; -1,5,-1; 0,-1,0]
   Output clamped to [0,255] to prevent overflow.
   =================================================================== */
__global__ void kernel_sharpen(
    const unsigned char* input,
    unsigned char* output,
    int width, int height)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= width || y >= height) return;

    const int K[3][3] = {{0,-1,0},{-1,5,-1},{0,-1,0}};
    float r=0.f, g=0.f, b=0.f;

    for (int ky=-1; ky<=1; ++ky) {
        for (int kx=-1; kx<=1; ++kx) {
            int nx = max(0, min(x+kx, width-1));
            int ny = max(0, min(y+ky, height-1));
            int src = (ny*width+nx)*3;
            int k = K[ky+1][kx+1];
            r += k * input[src+0];
            g += k * input[src+1];
            b += k * input[src+2];
        }
    }
    int dst = (y*width+x)*3;
    output[dst+0] = dev_clamp(r);
    output[dst+1] = dev_clamp(g);
    output[dst+2] = dev_clamp(b);
}

/* ===================================================================
   KERNEL 4: Gaussian Blur
   Kernel weights passed as a device array (d_kernel).
   Each thread accumulates a weighted sum over its neighbourhood.
   =================================================================== */
__global__ void kernel_gaussian_blur(
    const unsigned char* input,
    unsigned char* output,
    int width, int height,
    const float* d_kernel,
    int radius)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= width || y >= height) return;

    int ksize = 2*radius+1;
    float ar=0.f, ag=0.f, ab=0.f;

    for (int ky=-radius; ky<=radius; ++ky) {
        for (int kx=-radius; kx<=radius; ++kx) {
            int nx = max(0, min(x+kx, width-1));
            int ny = max(0, min(y+ky, height-1));
            int src = (ny*width+nx)*3;
            float w = d_kernel[(ky+radius)*ksize + (kx+radius)];
            ar += w * input[src+0];
            ag += w * input[src+1];
            ab += w * input[src+2];
        }
    }
    int dst = (y*width+x)*3;
    output[dst+0] = dev_clamp(ar);
    output[dst+1] = dev_clamp(ag);
    output[dst+2] = dev_clamp(ab);
}
"""

# Compile all kernels — nvcc runs here, errors appear immediately
mod = SourceModule(CUDA_SOURCE)

# Retrieve compiled kernel handles
k_grayscale = mod.get_function('kernel_grayscale')
k_sobel     = mod.get_function('kernel_sobel')
k_sharpen   = mod.get_function('kernel_sharpen')
k_gauss     = mod.get_function('kernel_gaussian_blur')

print('✅ All 4 CUDA kernels compiled successfully.')
print(f'   GPU: {cuda.Device(0).name()}')

## Cell 4 — GPU runner + timing helper

In [ ]:
"""
GPU runner functions.

Each function:
  1. Allocates GPU memory with cuda.mem_alloc()     [cudaMalloc equivalent]
  2. Copies input to GPU with cuda.memcpy_htod()    [cudaMemcpy HostToDevice]
  3. Configures 2D grid + block dimensions
  4. Records CUDA start event (GPU hardware timer)
  5. Launches the kernel via <<<grid, block>>> syntax
  6. Records CUDA stop event and synchronises
  7. Copies result back with cuda.memcpy_dtoh()     [cudaMemcpy DeviceToHost]
  8. Returns output image, kernel_ms, total_ms

Timing split:
  kernel_ms — pure GPU compute time (CUDA hardware events)
  total_ms  — kernel + memory transfers (host perf_counter)
"""

import time

def gpu_run(kernel_fn, input_arr, output_shape, block_size=16,
            extra_args=(), iterations=5, warmup=2):
    """
    Generic GPU runner with warm-up and multi-iteration averaging.

    Parameters
    ----------
    kernel_fn   : compiled PyCUDA kernel function
    input_arr   : contiguous uint8 NumPy array (host memory)
    output_shape: shape tuple for the output array
    block_size  : thread block side length (block is block_size x block_size)
    extra_args  : additional device pointers / scalars passed after output
    iterations  : number of timed runs (average is returned)
    warmup      : runs before timing starts (discarded)

    Returns
    -------
    output_arr : uint8 NumPy result array
    kernel_ms  : average GPU kernel time in milliseconds
    total_ms   : average end-to-end time (incl. memory transfers)
    """
    h, w = input_arr.shape[:2]
    img_c  = np.ascontiguousarray(input_arr, dtype=np.uint8)
    output = np.zeros(output_shape, dtype=np.uint8)

    # Allocate device memory (stays allocated across iterations for fair timing)
    d_input  = cuda.mem_alloc(img_c.nbytes)
    d_output = cuda.mem_alloc(output.nbytes)

    # Configure 2D grid: ceiling division so every pixel is covered
    block = (block_size, block_size, 1)
    grid  = ((w + block_size-1)//block_size,
             (h + block_size-1)//block_size, 1)

    def run_once():
        # HOST → DEVICE transfer
        cuda.memcpy_htod(d_input, img_c)
        # Kernel timing via CUDA hardware events
        ev_start = cuda.Event()
        ev_stop  = cuda.Event()
        ev_start.record()
        kernel_fn(d_input, d_output, np.int32(w), np.int32(h),
                  *extra_args, block=block, grid=grid)
        ev_stop.record()
        ev_stop.synchronize()          # wait for GPU to finish
        km = ev_start.time_till(ev_stop)
        # DEVICE → HOST transfer
        cuda.memcpy_dtoh(output, d_output)
        return km

    # Warm-up iterations (CUDA context + cache priming)
    for _ in range(warmup):
        run_once()

    # Measured iterations
    k_times, t_times = [], []
    for _ in range(iterations):
        t0 = time.perf_counter()
        km = run_once()
        t_times.append((time.perf_counter()-t0)*1000)
        k_times.append(km)

    return output.copy(), sum(k_times)/len(k_times), sum(t_times)/len(t_times)


def build_gaussian_kernel(radius, sigma):
    """Build a normalised Gaussian kernel and upload it to GPU memory."""
    ksize = 2*radius+1
    k = np.zeros((ksize, ksize), dtype=np.float32)
    for ky in range(-radius, radius+1):
        for kx in range(-radius, radius+1):
            k[ky+radius, kx+radius] = np.exp(-(kx**2+ky**2)/(2*sigma**2))
    k /= k.sum()  # normalise
    d_k = cuda.mem_alloc(k.nbytes)
    cuda.memcpy_htod(d_k, np.ascontiguousarray(k))
    return d_k

print('✅ GPU runner functions defined.')

## Cell 5 — Create or upload a test image

In [ ]:
"""
Image setup.

Option A: Use the synthetic test image generated below (no upload needed).
Option B: Upload your own image by uncommenting the file upload code.

We test on multiple resolutions so you can see how speedup scales with image size.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

# ── Option A: Synthetic test image ────────────────────────────────────────────
def make_test_image(size=512):
    """Generate a colourful synthetic image with edges for good Sobel results."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    # Background gradient
    for y in range(size):
        img[y, :, 2] = int(40 + y * 180 / size)
    # Bright circle
    cx, cy = size//3, size//3
    r = size//5
    Y, X = np.ogrid[:size, :size]
    mask = (X-cx)**2 + (Y-cy)**2 <= r**2
    img[mask] = [220, 60, 60]
    # Green rectangle
    s4, s2 = size//4, size//2
    img[s2:s2+s4, s4:s4+s2, :] = [60, 200, 80]
    # White stripe (creates strong edges for Sobel)
    for i in range(size):
        for d in range(-3, 4):
            j = i + d
            if 0 <= j < size:
                img[i, j] = [240, 240, 240]
    return img

img_512  = make_test_image(512)
img_1080 = make_test_image(1080)

# ── Option B: Upload your own image (uncomment below) ─────────────────────────
# from google.colab import files
# uploaded = files.upload()
# fname = list(uploaded.keys())[0]
# img_512 = cv2.cvtColor(cv2.imread(fname), cv2.COLOR_BGR2RGB)
# img_512 = cv2.resize(img_512, (512, 512))
# img_1080 = cv2.resize(img_512, (1920, 1080))

print(f'Test image 512:  {img_512.shape}  {img_512.dtype}')
print(f'Test image 1080: {img_1080.shape}  {img_1080.dtype}')

plt.figure(figsize=(5,5))
plt.imshow(img_512)
plt.title('Test Image (512×512)')
plt.axis('off')
plt.tight_layout()
plt.show()

## Cell 6 — Run all operations and collect benchmark results

In [ ]:
"""
Run all four operations on both image sizes.
For each operation: CPU time, GPU kernel time, GPU total time, and speedup.
"""

BLOCK_SIZE = 16   # 16×16 = 256 threads per block
ITERATIONS = 5
WARMUP     = 2

results = {}  # store all benchmark results

for img, label in [(img_512, '512×512'), (img_1080, '1920×1080')]:
    h, w = img.shape[:2]
    gray = cpu_grayscale(img)  # grayscale input for Sobel
    results[label] = {}

    print(f'\n{'='*55}')
    print(f'  Image: {label}  ({w*h:,} pixels)')
    print(f'  Block: {BLOCK_SIZE}×{BLOCK_SIZE} = {BLOCK_SIZE**2} threads/block')
    print(f'{'='*55}')

    # ── 1. Grayscale ──────────────────────────────────────────────────────────
    cpu_out, cpu_ms = time_cpu(cpu_grayscale, img, iterations=ITERATIONS, warmup=WARMUP)
    gpu_out, k_ms, t_ms = gpu_run(
        k_grayscale, img, (h, w),
        block_size=BLOCK_SIZE, iterations=ITERATIONS, warmup=WARMUP)
    speedup = cpu_ms / k_ms
    results[label]['Grayscale'] = dict(cpu=cpu_ms, kernel=k_ms, total=t_ms, speedup=speedup)
    print(f'  Grayscale   CPU:{cpu_ms:7.2f}ms  GPU kernel:{k_ms:6.3f}ms  Total:{t_ms:7.2f}ms  Speedup:{speedup:5.1f}×')

    # ── 2. Gaussian Blur ──────────────────────────────────────────────────────
    RADIUS, SIGMA = 3, 1.5
    d_kern = build_gaussian_kernel(RADIUS, SIGMA)
    cpu_out, cpu_ms = time_cpu(cpu_gaussian_blur, img, RADIUS, SIGMA, iterations=ITERATIONS, warmup=WARMUP)
    gpu_out, k_ms, t_ms = gpu_run(
        k_gauss, img, img.shape,
        block_size=BLOCK_SIZE,
        extra_args=(d_kern, np.int32(RADIUS)),
        iterations=ITERATIONS, warmup=WARMUP)
    speedup = cpu_ms / k_ms
    results[label]['Gaussian Blur'] = dict(cpu=cpu_ms, kernel=k_ms, total=t_ms, speedup=speedup)
    print(f'  Gauss Blur  CPU:{cpu_ms:7.2f}ms  GPU kernel:{k_ms:6.3f}ms  Total:{t_ms:7.2f}ms  Speedup:{speedup:5.1f}×')

    # ── 3. Sobel Edge Detection ───────────────────────────────────────────────
    cpu_out, cpu_ms = time_cpu(cpu_sobel, gray, iterations=ITERATIONS, warmup=WARMUP)
    gpu_out, k_ms, t_ms = gpu_run(
        k_sobel, gray, (h, w),
        block_size=BLOCK_SIZE, iterations=ITERATIONS, warmup=WARMUP)
    speedup = cpu_ms / k_ms
    results[label]['Sobel'] = dict(cpu=cpu_ms, kernel=k_ms, total=t_ms, speedup=speedup)
    print(f'  Sobel       CPU:{cpu_ms:7.2f}ms  GPU kernel:{k_ms:6.3f}ms  Total:{t_ms:7.2f}ms  Speedup:{speedup:5.1f}×')

    # ── 4. Sharpening ─────────────────────────────────────────────────────────
    cpu_out, cpu_ms = time_cpu(cpu_sharpen, img, iterations=ITERATIONS, warmup=WARMUP)
    gpu_out, k_ms, t_ms = gpu_run(
        k_sharpen, img, img.shape,
        block_size=BLOCK_SIZE, iterations=ITERATIONS, warmup=WARMUP)
    speedup = cpu_ms / k_ms
    results[label]['Sharpening'] = dict(cpu=cpu_ms, kernel=k_ms, total=t_ms, speedup=speedup)
    print(f'  Sharpen     CPU:{cpu_ms:7.2f}ms  GPU kernel:{k_ms:6.3f}ms  Total:{t_ms:7.2f}ms  Speedup:{speedup:5.1f}×')

print('\n✅ Benchmarks complete.')

## Cell 7 — Visualise processed images side by side

In [ ]:
"""
Show original + all four GPU-processed outputs in a grid.
"""

import matplotlib.pyplot as plt
import numpy as np

img = img_512
gray = cpu_grayscale(img)
BLOCK_SIZE = 16

# Generate final GPU outputs for display
gray_gpu,  _, _ = gpu_run(k_grayscale, img,  (img.shape[0], img.shape[1]), BLOCK_SIZE, iterations=1, warmup=0)
d_kern = build_gaussian_kernel(3, 1.5)
blur_gpu,  _, _ = gpu_run(k_gauss, img, img.shape, BLOCK_SIZE,
                           extra_args=(d_kern, np.int32(3)), iterations=1, warmup=0)
sobel_gpu, _, _ = gpu_run(k_sobel, gray, (img.shape[0], img.shape[1]), BLOCK_SIZE, iterations=1, warmup=0)
sharp_gpu, _, _ = gpu_run(k_sharpen, img, img.shape, BLOCK_SIZE, iterations=1, warmup=0)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
images = [
    (img,       'Original',        'viridis'),
    (gray_gpu,  'Grayscale (GPU)', 'gray'),
    (blur_gpu,  'Gaussian Blur (GPU)', None),
    (sobel_gpu, 'Sobel Edges (GPU)', 'gray'),
    (sharp_gpu, 'Sharpened (GPU)', None),
]
for ax, (im, title, cmap) in zip(axes, images):
    kwargs = {'cmap': cmap} if cmap else {}
    ax.imshow(im, **kwargs)
    ax.set_title(title, fontsize=10, pad=6)
    ax.axis('off')

plt.suptitle('GPU Image Processing Results (512×512)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('gpu_results.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Saved to gpu_results.png')

## Cell 8 — Performance comparison charts

In [ ]:
"""
Generate a professional benchmark chart:
  - Left:  CPU vs GPU execution times per operation
  - Right: Speedup (CPU time / GPU kernel time)
"""

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

COLORS = {'CPU': '#3b82d4', 'GPU Kernel': '#7c5cd8', 'GPU Total': '#10b981'}
ops = ['Grayscale', 'Gaussian Blur', 'Sobel', 'Sharpening']
x = np.arange(len(ops))
W = 0.25

for col, (label, res) in enumerate(results.items()):
    ax = axes[col]
    cpu_t    = [res[op]['cpu']    for op in ops]
    kern_t   = [res[op]['kernel'] for op in ops]
    total_t  = [res[op]['total']  for op in ops]
    speedups = [res[op]['speedup'] for op in ops]

    if col == 0:
        # Left chart: execution time bars
        b1 = ax.bar(x - W,   cpu_t,   W, label='CPU',        color=COLORS['CPU'],        edgecolor='none')
        b2 = ax.bar(x,       kern_t,  W, label='GPU Kernel', color=COLORS['GPU Kernel'], edgecolor='none')
        b3 = ax.bar(x + W,   total_t, W, label='GPU Total',  color=COLORS['GPU Total'],  edgecolor='none')
        ax.set_ylabel('Execution Time (ms)', fontsize=11)
        ax.set_title(f'Execution Time — {label}', fontsize=12)
        ax.legend(fontsize=9)
        for bars in [b1, b2, b3]:
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x()+bar.get_width()/2, h+max(cpu_t)*0.01,
                        f'{h:.1f}', ha='center', va='bottom', fontsize=7)
    else:
        # Right chart: speedup bars
        bars = ax.bar(x, speedups, 0.5, color=COLORS['GPU Kernel'], edgecolor='none')
        ax.axhline(y=1, color='#e5e7eb', linestyle='--', linewidth=1)
        ax.set_ylabel('Speedup (CPU time / GPU kernel time)', fontsize=11)
        ax.set_title(f'GPU Speedup — {label}', fontsize=12)
        for bar, s in zip(bars, speedups):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                    f'{s:.1f}×', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(ops, fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('CPU vs GPU Performance — Real Measurements on Google Colab T4', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('benchmark_chart.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Saved to benchmark_chart.png')

## Cell 9 — Block size comparison (8×8 vs 16×16 vs 32×32)

In [ ]:
"""
Compare how CUDA block size affects GPU kernel performance.

Block size affects GPU occupancy — how well the GPU's thread schedulers
can keep all streaming multiprocessors busy.
  - Too small (8×8 = 64 threads): low occupancy, underutilises GPU.
  - Optimal (16×16 = 256 threads): good balance for most GPUs.
  - Large (32×32 = 1024 threads): maximum threads/block, may hit register limits.
"""

import matplotlib.pyplot as plt
import numpy as np

block_sizes = [8, 16, 32]
block_results = {}

for bsz in block_sizes:
    _, k_ms, _ = gpu_run(k_grayscale, img_512, (img_512.shape[0], img_512.shape[1]),
                          block_size=bsz, iterations=10, warmup=3)
    block_results[f'{bsz}×{bsz}\n({bsz**2} threads)'] = k_ms
    print(f'  Block {bsz}×{bsz} ({bsz**2:4d} threads/block): {k_ms:.3f} ms')

fig, ax = plt.subplots(figsize=(7, 4))
labels = list(block_results.keys())
times  = list(block_results.values())
bars = ax.bar(labels, times, color=['#3b82d4','#7c5cd8','#10b981'], width=0.4, edgecolor='none')
for bar, t in zip(bars, times):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(times)*0.01,
            f'{t:.3f} ms', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Grayscale Kernel Time vs CUDA Block Size (512×512 image)', fontsize=12)
ax.set_ylabel('Kernel Time (ms)', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('block_size_chart.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Saved to block_size_chart.png')

## Cell 10 — CPU vs GPU numerical correctness check

In [ ]:
"""
Verify that GPU kernel outputs match CPU reference outputs within
a small numerical tolerance (floating-point rounding differences are expected).
"""

import numpy as np

TOLERANCE = 2  # max allowed pixel difference

def check_consistency(op_name, cpu_out, gpu_out, tol=TOLERANCE):
    diff = np.abs(cpu_out.astype(np.int32) - gpu_out.astype(np.int32))
    max_d  = diff.max()
    mean_d = diff.mean()
    match  = '✅ PASS' if max_d <= tol else '❌ FAIL'
    print(f'  {op_name:<20} max_diff={max_d:3d}  mean_diff={mean_d:.3f}  [{match}]')
    return max_d <= tol

img   = img_512
gray  = cpu_grayscale(img)
BLOCK = 16

print(f'Tolerance: ±{TOLERANCE} per pixel\n')

# Grayscale
cpu_g = cpu_grayscale(img)
gpu_g, _, _ = gpu_run(k_grayscale, img, (img.shape[0], img.shape[1]), BLOCK, iterations=1, warmup=0)
check_consistency('Grayscale', cpu_g, gpu_g)

# Sobel
cpu_s = cpu_sobel(gray)
gpu_s, _, _ = gpu_run(k_sobel, gray, gray.shape, BLOCK, iterations=1, warmup=0)
check_consistency('Sobel', cpu_s, gpu_s)

# Sharpening
cpu_sh = cpu_sharpen(img)
gpu_sh, _, _ = gpu_run(k_sharpen, img, img.shape, BLOCK, iterations=1, warmup=0)
check_consistency('Sharpening', cpu_sh, gpu_sh, tol=3)

# Gaussian Blur
cpu_bl = cpu_gaussian_blur(img, 3, 1.5)
d_k = build_gaussian_kernel(3, 1.5)
gpu_bl, _, _ = gpu_run(k_gauss, img, img.shape, BLOCK, extra_args=(d_k, np.int32(3)), iterations=1, warmup=0)
check_consistency('Gaussian Blur', cpu_bl, gpu_bl, tol=2)

print('\nSmall differences are expected due to floating-point rounding on GPU vs CPU.')

## Cell 11 — Final summary table

In [ ]:
"""
Print a clean summary of all benchmark results.
These are real measurements on the assigned Colab GPU — nothing fabricated.
"""

import pandas as pd

rows = []
for res_label, res in results.items():
    for op, data in res.items():
        rows.append({
            'Resolution'    : res_label,
            'Operation'     : op,
            'CPU Time (ms)' : f"{data['cpu']:.2f}",
            'GPU Kernel (ms)': f"{data['kernel']:.3f}",
            'GPU Total (ms)' : f"{data['total']:.2f}",
            'Speedup'       : f"{data['speedup']:.1f}×",
            'Improvement'   : f"{(data['speedup']-1)*100:.0f}%" if data['speedup'] > 1 else '—',
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print(f'\nGPU: {cuda.Device(0).name()}')
print('Timing: CPU = time.perf_counter() | GPU kernel = CUDA hardware events')
print('Warm-up: 2 iterations discarded | Measured: 5 iterations averaged')
print('Speedup = CPU_avg_ms / GPU_kernel_avg_ms')

## Cell 12 — Download result images
Run this cell to download the saved charts to your computer.

In [ ]:
from google.colab import files
import os

for fname in ['gpu_results.png', 'benchmark_chart.png', 'block_size_chart.png']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloaded: {fname}')
    else:
        print(f'Not found (run cells 7-9 first): {fname}')